# 02 — Progression S1 → S2 — PRIMARY (patellofemoral) ⭐

**Notebook id**: `02_progression_total` → vault `03.3-progression-S1-S2.md`

**Primary estimand** (consensus point B): the **patellofemoral block** Δ`lesion_pf` = {trochlée, rotule} progresses more in Cyclops than in Meniscus.

- **Frequentist support**: Mann–Whitney U + **Cliff's δ** (rank-based, no interval assumption) + **exact/Monte-Carlo permutation** p + **BCa** CI (B = 10 000) + a **test-inversion** guard-rail CI — all via `tf.pf_contrast`.
- **Decision rule** (point G): the global Bayesian verdict lives in M3 (notebook 05); the permutation p is the frequentist support, not a separate gate.
- **Binary effect**: the worsened-PF odds ratio is estimated with **Firth penalised logistic** (`tf.firth_or`) — crude **and** sex+age-adjusted. The 1/20 meniscus events near-separate the plain ML logit into a *ghost* OR≈24 (≈47 adjusted); Firth gives a stable OR + CI, and we report an **E-value** (`tf.evalue_or`) for robustness to unmeasured confounding.
- **Demoted / descriptive**: the 6-compartment sum Δ`lesion_total` is a *global burden index* that **dilutes** the compartmental signal — reported (two- and one-sided) but **not** decisional.
- **M2** (NegBin on the 6-sum, Δ⁺ truncation): **removed** from the inferential chain.

In [ ]:
import sys
from pathlib import Path

current = Path().absolute().parent
sys.path.insert(0, (current / "src").as_posix())


In [ ]:
# --- Setup (idempotent, fresh-kernel reproducible) ---
import warnings

warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

from constants import (
    RANDOM_SEED,
    SITES,
    SITES_PF,
    SITES_FT,
    SITES_BINARY,
    SITES_ORDINAL,
    GROUPS,
    BLOCKS,
    N_TOTAL,
    N_TOTAL_ANALYSABLE,
    N_MENISCUS,
    N_CYCLOPS,
    SCORE_MAX,
    SCORE_MAX_COLLAPSED,
)
import loaders
import preprocessing as pp
import tests_freq as tf
import reporting as rpt
import bayes_models as bm
import viz

np.random.seed(RANDOM_SEED)
viz.set_pub_style()


In [ ]:
# --- Load & preprocess (canonical pipeline) ---
df = loaders.load_combined()
df = pp.apply_date_hygiene(df)  # composite-key (group, anonyme) date hygiene
df = pp.add_derived(df)  # lesion_pf/ft, female, deltas, worsened_pf, ...
wide = pp.to_wide(df)  # one row per patient (group, anonyme)
patient = pp.to_patient(df)  # static covariates per patient

# Patient-level covariates joined onto the wide outcomes (for H3 / sensitivity).
_cov = [
    c
    for c in [
        "group",
        "anonyme",
        "female",
        "sexe",
        "pivot_pivot_contact",
        "travail_physique",
        "tabac",
        "age_at_trauma",
        "imc",
        "taille",
        "poids",
    ]
    if c in patient.columns
]
merged = wide.merge(patient[_cov], on=["group", "anonyme"], how="left")

print(
    "long:",
    df.shape,
    "| wide:",
    wide.shape,
    "| patient:",
    patient.shape,
    "| merged:",
    merged.shape,
)
# Composite-key sentinel: 19 Anonyme ids are reused across the two sheets.
assert (df.groupby(["group", "anonyme"]).size() == 2).all(), "composite key broken"


## 1. Distribution of the primary outcome Δ`lesion_pf`

In [ ]:
delta_pf = wide[["group", "delta_lesion_pf"]].dropna()
print(delta_pf.groupby("group")["delta_lesion_pf"].describe())
print()
worsened = wide.groupby("group")["worsened_pf"].apply(
    lambda s: (s.dropna() == 1).mean()
)
print("Proportion worsening (Δ_PF > 0) by group:")
print((worsened * 100).round(1).astype(str) + " %")


## 2. PRIMARY contrast — `tf.pf_contrast` (MWU + Cliff δ + permutation + BCa + inversion)

In [ ]:
pf = tf.pf_contrast(
    wide, value_col="delta_lesion_pf", n_boot=10000, n_perm=20000, seed=RANDOM_SEED
)
for k, v in pf.items():
    print(f"  {k:26s} = {v}")

print()
print("--- verdict (frequentist support) ---")
print(
    rpt.verdict_freq_en(
        pf["perm_p"],
        "Primary PF contrast (cyclops vs meniscus)",
        effect=f"Cliff delta = {pf['cliffs_delta']:+.2f} ({pf['cliffs_delta_magnitude']})",
    )
)


## 3. PRIMARY figure — raincloud of Δ`lesion_pf` by group

In [ ]:
worsened_pct = {
    g: float((wide.loc[wide.group == g, "delta_lesion_pf"].dropna() > 0).mean() * 100)
    for g in GROUPS
}
fig = viz.raincloud_progression(
    wide,
    value_col="delta_lesion_pf",
    worsened_pct=worsened_pct,
    cliff_delta=pf["cliffs_delta"],
    perm_p=pf["perm_p"],
    title="Patellofemoral progression S1 -> S2 (primary)",
    ylabel="Δ patellofemoral lesion score (S2 - S1)",
)
fig


## 4. PF score trajectories S1 → S2 (slopegraph)

In [ ]:
fig = viz.slopegraph_pf_mpl(wide)
fig


## 4b. Binary worsened-PF odds ratio — Firth penalised logistic + E-value

Dichotomising the PF outcome (`worsened_pf` = Δ`lesion_pf` > 0) gives a 2×2 with only **1/19** worsening events in meniscus → near-complete separation. Plain ML logistic returns a *ghost* OR ≈ 24 (≈ 47 once adjusted) with an explosive CI — it must **not** anchor the paper. `tf.firth_or` (Firth/Jeffreys penalised likelihood) returns a finite, stable OR. We report:

- **crude** OR (group only);
- **sex + age-adjusted** OR (`covariates=('female','age_at_trauma')`) — the headline robustness analysis: the PF effect survives adjustment;
- an **E-value** (`tf.evalue_or`, common-outcome `RR≈√OR`): the minimum association an unmeasured confounder would need with *both* group and outcome to explain the effect away (large ⇒ robust).

In [ ]:
or_crude = tf.firth_or(wide, outcome_col="worsened_pf", covariates=())
or_adj = tf.firth_or(
    merged, outcome_col="worsened_pf", covariates=("female", "age_at_trauma")
)
print("Firth penalised-logistic OR for worsened_pf (cyclops vs meniscus):")
print(f"  method        : {or_adj['method']}")
print(
    f"  separation_ml : {or_crude['separation_ml']}  (min 2x2 cell = {or_crude['min_cell']})"
)
print(
    f"  events        : cases {or_crude['n_events_case']}/{'?'}  "
    f"controls {or_crude['n_events_control']} (the 1-event meniscus cell)"
)
print()
print(
    f"  crude        : OR = {or_crude['odds_ratio']:.2f}  "
    f"[{or_crude['or_ci_lo']:.2f}, {or_crude['or_ci_hi']:.2f}]  "
    f"p = {or_crude['p']:.4f}  n = {or_crude['n']}"
)
print(
    f"  sex+age adj. : OR = {or_adj['odds_ratio']:.2f}  "
    f"[{or_adj['or_ci_lo']:.2f}, {or_adj['or_ci_hi']:.2f}]  "
    f"p = {or_adj['p']:.4f}  n = {or_adj['n']}  cov={or_adj['covariates']}"
)
print("  (the effect SURVIVES sex+age adjustment; the plain-ML OR=24/47 was a ghost)")


In [ ]:
# E-value for the crude Firth OR (worsened_pf is a common outcome -> RR ~ sqrt(OR)).
ev = tf.evalue_or(
    or_crude["odds_ratio"], or_ci_lo=or_crude["or_ci_lo"], common_outcome=True
)
print("E-value for the crude worsened-PF OR (VanderWeele & Ding 2017):")
print(f"  RR approx    : {ev['rr_approx']:.2f}  ({ev['method']})")
print(f"  E-value point: {ev['evalue_point']:.2f}")
print(f"  E-value CI   : {ev['evalue_ci']}")
print("  Interpretation: an unmeasured confounder would need associations of at")
print("  least this strength with BOTH group and worsened_pf to explain it away.")


## 5. DESCRIPTIVE / SECONDARY — 6-compartment sum (dilution, non-decisional)

The 6-sum mixes commensurable PF grades with rare/absent FT events; it **dilutes** the signal to a small effect. We report it transparently with both the two-sided p (transparency) and the one-sided p (direction was pre-registered) — **neither drives any decision** (point G).

In [ ]:
s6 = wide[["group", "delta_lesion_total"]].dropna()
c6 = s6.loc[s6.group == "cyclops", "delta_lesion_total"].astype(float).values
m6 = s6.loc[s6.group == "meniscus", "delta_lesion_total"].astype(float).values
from scipy import stats as _st

res6 = tf.mwu_with_effects(c6, m6, n_boot=10000, seed=RANDOM_SEED)
_, p_one = _st.mannwhitneyu(c6, m6, alternative="greater")
print(
    f"6-sum Cliff delta = {res6['cliffs_delta']:+.3f} ({res6['cliffs_delta_magnitude']})"
)
print(f"6-sum MWU p two-sided = {res6['pvalue']:.4f}   one-sided = {p_one:.4f}")
print(f"BCa 95% CI = [{res6['delta_ci_lo']:+.3f}, {res6['delta_ci_hi']:+.3f}]")
print()
print(
    "Interpretation: the global burden index understates a compartment-specific"
    " effect (PF). It is descriptive only; the primary inference is the PF contrast."
)


## 6. M2 — removed from the inferential chain (note only)

The earlier M2 (Negative-Binomial on the 6-compartment sum with a Δ⁺ = max(Δ, 0) truncation) is **removed** (consensus point D): summing non-commensurable ordinals and discarding improvements is incoherent. A labelled NegBin **sanity-check** on the post-op PF score adjusted for baseline (no truncation) is available as `bm.fit_m2_negbin_sanity`, but it is **not** part of the primary inference (which is M3 + permutation).

In [ ]:
# Optional labelled sanity-check (NOT primary; no Δ⁺ truncation). Uncomment to run.
sane = bm.fit_m2_negbin_sanity(
    wide, outcome_col="lesion_pf_S2", baseline_col="lesion_pf_S1"
)
print(sane["note"])
print(rpt.summary_bayes(sane["idata"]))
print("M2 is removed from the primary chain (see markdown above).")


## Sanity asserts

In [ ]:
assert wide["delta_lesion_pf"].notna().sum() == N_TOTAL_ANALYSABLE == 69
assert pf["cliffs_delta"] > 0 and pf["perm_p"] < 0.01
print("Primary PF endpoint asserts passed.")
